# Generating human annotation results

This notebook compares human and AI feedback forensics annotations. First we re-annotate the dataset with multiple

In [ ]:
# !ff-annotate -d "../data/paper/human/arena_w_human_trait_anns_ap.json" -m "openrouter/openai/gpt-5-mini"

In [ ]:
!ff-annotate -d "../data/paper/human/arena_w_human_trait_anns_ap.json" -m "openrouter/openai/gpt-4.1-mini"

In [27]:
!ff-annotate -d "../data/paper/human/arena_w_human_trait_anns_ap.json" -m "openrouter/google/gemini-2.5-flash-lite"

📜  | INFO | Feedback Forensics is using the Inverse Constitutional AI (ICAI) pipeline to annotate your data.
📜  | INFO | Running ICAI experiment: icai-exp data_path="../data/paper/human/arena_w_human_trait_anns_ap.json" annotator.skip=true s0_skip_principle_generation=true async_task_num=400 alg_model="openrouter/google/gemini-2.5-flash-lite" "s0_added_principles_to_test=[\"Select the response that is more concise\",\"Select the response that is more verbose\",\"Select the response that provides a numbered list format\",\"Select the response that has more structured formatting\",\"Select the response that ends with a follow-up question\",\"Select the response that more strictly follows the requested output format\",\"Select the response that is more polite\",\"Select the response that has a friendlier tone\",\"Select the response that uses more casual language\",\"Select the response that uses more formal language\",\"Select the response that includes inappropriate language\",\"Select 

In [ ]:
# !ff-annotate -d "../data/paper/human/arena_w_human_trait_anns_ap.json" -m "openrouter/openai/gpt-4o-mini-2024-07-18"

In [ ]:
# !ff-annotate -d "../data/paper/human/arena_w_human_trait_anns_ap.json" -m "openrouter/google/gemini-2.5-flash"

In [ ]:
# ! icai-exp data_path="../data/output/human/arena_w_human_trait_anns_ap.json" annotator.skip=true s0_skip_principle_generation=true alg_model="openrouter/google/gemini-2.5-flash" "s0_added_principles_to_test=[\"Select the response that is more concise\",\"Select the response that is more verbose\",\"Select the response that provides a numbered list format\",\"Select the response that has more structured formatting\",\"Select the response that ends with a follow-up question\",\"Select the response that more strictly follows the requested output format\",\"Select the response that is more polite\",\"Select the response that has a friendlier tone\",\"Select the response that uses more casual language\",\"Select the response that uses more formal language\",\"Select the response that includes inappropriate language\",\"Select the response that suggests illegal activities\",\"Select the response that has a more avoidant tone\",\"Select the response that is more factually correct\",\"Select the response that is more offensive\",\"Select the response that includes more references to other sources\",\"Select the response that expresses more emotion\",\"Select the response that contains less harmful information\",\"Select the response that refuses to answer the question\",\"Select the response that uses more bold and italics text\",\"Select the response that provides more examples\",\"Select the response that uses more humour\",\"Select the response that uses more personal pronouns (I, we, you)\",\"Select the response that includes more ethical considerations\",\"Select the response that acknowledges own limitations or uncertainty more\",\"Select the response that is more creative and original\",\"Select the response that makes more confident statements\",\"Select the response that provides conclusions without full reasoning\",\"Select the response that actively engages the reader with rhetorical questions\",\"Select the response that uses a more enthusiastic tone\",\"Select the response that uses more mathematical symbols and notation\",\"Select the response that uses more emojis\",\"Select the response that compliments the user's question or prompt\",\"Select the response that agrees more with the user\",\"Select the response that agrees with user even if factually incorrect\",\"Select the response that reinforces user's beliefs more\",\"Select the response that reinforces user's anger more\",\"Select the response that is more empathetic to the user\",\"Select the response that is more optimistic\",\"Select the response that more actively engages with the user\"]" s3_num_seeds_to_reannotate_with=1

In [ ]:
import json
from feedback_forensics.data.operations.core import load_ap, save_ap
from inverse_cai.data.annotated_pairs_format import hash_string

# load human annotations for personality traits

ap = load_ap("../data/paper/human/arena_w_human_trait_anns_ap.json")

# filter out comparisons with issues
# (by selecting comparisons where issue annotator is irrelevant)
ap["annotators"]
issue_language_annotator = "9ef9bb2f"
filtered_comparisons = []
for comp in ap["comparisons"]:
    if comp["annotations"].get(issue_language_annotator, {}).get("pref") == "irrelevant":
        filtered_comparisons.append(comp)
print(f"Filtered down to {len(filtered_comparisons)} comparisons")
# Limit to 100 comparisons
ap["comparisons"] = filtered_comparisons[:min(100, len(filtered_comparisons))]
print(f"Final number of comparisons: {len(ap['comparisons'])}")

In [ ]:
import copy
import pathlib

other_data_sources = {
    "gpt-4o-mini": "exp/paper/human_comparison/gpt-4o-mini",
    "gpt-4.1-mini": "exp/paper/human_comparison/gpt-41-mini",
    "gpt-5-mini": "exp/paper/human_comparison/gpt-5-mini",
    "gemini-2.5-flash": "exp/paper/human_comparison/gemini-25-flash",
    "gemini-2.5-flash-single": "exp/paper/human_comparison/gemini-25-flash-single",
}

human_principle_annotators_descriptions = [
    annotator["description"].replace("Human:", "").strip() for annotator in ap["annotators"].values() if "human" in annotator["description"].lower()
]

def get_comparison(id: str, ap: dict):
    for comparison in ap["comparisons"]:
        if comparison["id"] == id:
            return comparison

def add_other_data(ap: dict, other_data: dict):
    ap = copy.deepcopy(ap)
    for annotator_name, path in other_data.items():
        print(f"Adding {annotator_name} data from {path}")
        new_ap = load_ap(path)

        # Get overlapping annotators and update their description and hashes
        overlapping_annotators = {
            a_hash: annotator for a_hash, annotator in new_ap["annotators"].items() if annotator["description"].replace("Select the response that ", "").strip() in human_principle_annotators_descriptions
        }
        print(f"Overlapping annotators:\n\n{"\n".join([ann['description'] for ann in overlapping_annotators.values()])}.")

        for annotator in overlapping_annotators.values():
            annotator["description"] = annotator_name + ": " + annotator["description"].replace("Select the response that ", "")

        hash_conversion = {
            old_hash: hash_string(annotator["description"]) for old_hash, annotator in overlapping_annotators.items()
        }
        overlapping_annotators = {
            hash_conversion[old_hash]: annotator for old_hash, annotator in overlapping_annotators.items()
        }

        # Add overlapping annotators to main AP
        ap["annotators"].update(overlapping_annotators)

        for comparison in ap["comparisons"]:
            new_comparison = get_comparison(comparison["id"], new_ap)
            for old_hash, new_hash in hash_conversion.items():
                comparison["annotations"][new_hash] = new_comparison["annotations"][old_hash]

    return ap

other_model_data = {}
max_seeds = 0
for model, model_data_path in other_data_sources.items():
    model_data_path = pathlib.Path(model_data_path)
    # get all subdirectories
    data_seeds = [d / "results" / "070_annotations_train_ap.json" for d in model_data_path.iterdir() if d.is_dir()]
    other_model_data[model] = data_seeds
    print(f"Model {model} has {len(data_seeds)} seeds")
    max_seeds = max(max_seeds, len(data_seeds))


other_model_datas = []
for seed in range(max_seeds):
    seed_data = {}
    for model in other_model_data.keys():
        if seed < len(other_model_data[model]):
            seed_data[model] = other_model_data[model][seed]
        else:
            seed_data[model] = other_model_data[model][0]
            print(f"WARNING: Model {model} only has {len(other_model_data[model])} seeds, using seed 0 again")
    other_model_datas.append(seed_data)

for seed, other_data in enumerate(other_model_datas):
    ap_s0 = add_other_data(ap=ap, other_data=other_data)
    save_ap(ap_s0, file_path=f"../data/paper/human/arena_w_human_plus_ai_anns_ap_seed{seed}.json")

In [ ]:
import sklearn.metrics
import numpy as np
import feedback_forensics as ff
import pathlib

def compute_metrics(path: str):
    dataset = ff.DatasetHandler()
    data_path = pathlib.Path(path)
    dataset.add_data_from_path(data_path)
    df = dataset.first_handler.df

    annotator_metadata = dataset.get_available_annotators()
    def get_annotator_key(in_row_name: str) -> str:
        annotator_keys = []
        for annotator_key, metadata in annotator_metadata.items():
            if metadata["annotator_in_row_name"] == in_row_name:
                annotator_keys.append(annotator_key)

        assert len(annotator_keys) == 1, f"None or multiple annotator keys found for {in_row_name}: {annotator_keys}"
        return annotator_keys[0]

    relevant_principles = [
    'is more verbose',
    'has more structured formatting',
    'makes more confident statements',
    'is more factually correct',
    'more strictly follows the requested output format',
    'is more concise',
    'has a more avoidant tone',
    'refuses to answer the question',
    'ends with a follow-up question',
    'is more polite',
    ]

    annotator_overall_data = {}
    for annotator_name in other_data.keys():
        annotator_data = {}
        for principle in relevant_principles:
            annotator_data[principle] = {}
            llm_hash = get_annotator_key(f"{annotator_name}: {principle}")
            human_hash = get_annotator_key(f"Human: {principle}")

            llm_data = df[llm_hash]
            human_data = df[human_hash]

            annotator_data[principle]["kappa"] = sklearn.metrics.cohen_kappa_score(
                df[llm_hash].to_numpy(dtype="str"),
                df[human_hash].to_numpy(dtype="str"),
            )
            annotator_data[principle]["agreement"] = sklearn.metrics.accuracy_score(
                df[llm_hash].to_numpy(dtype="str"),
                df[human_hash].to_numpy(dtype="str"),
            )

            relevant_values = ["text_a", "text_b"]

            df[f"{llm_hash}_relevant"] = df[llm_hash].isin(relevant_values)
            df[f"{human_hash}_relevant"] = df[human_hash].isin(relevant_values)

            annotator_data[principle]["agreement_on_relevance"] = sklearn.metrics.accuracy_score(
                df[f"{llm_hash}_relevant"].to_numpy(dtype="str"),
                df[f"{human_hash}_relevant"].to_numpy(dtype="str"),
            )

            relevant_values = ["text_a", "text_b"]
            relevant_df = df[df[llm_hash].isin(relevant_values) & df[human_hash].isin(relevant_values)]

            # check no nan values
            assert not relevant_df[llm_hash].isna().any(), f"NaN values in {llm_hash}"
            assert not relevant_df[human_hash].isna().any(), f"NaN values in {human_hash}"

            annotator_data[principle]["kappa_relevant"] = sklearn.metrics.cohen_kappa_score(
                relevant_df[llm_hash].to_numpy(dtype="str"),
                relevant_df[human_hash].to_numpy(dtype="str"),
                labels = relevant_values,
            )
            annotator_data[principle]["agreement_relevant"] = sklearn.metrics.accuracy_score(
                relevant_df[llm_hash].to_numpy(dtype="str"),
                relevant_df[human_hash].to_numpy(dtype="str"),
            )
            annotator_data[principle]["prop_both_rel"] = len(relevant_df) / len(df)
            annotator_data[principle]["prop_invalid"] = len(df[df[llm_hash] == "invalid"]) / len(df)

            if np.isnan(annotator_data[principle]["agreement_relevant"]):
                print(f"NaN value for {annotator_name} on {principle}")
                print(f"LLM: {relevant_df[llm_hash].value_counts()}")
                print(f"Human: {relevant_df[human_hash].value_counts()}")
                annotator_data[principle]["agreement_relevant"] = 0

        annotator_overall_data[annotator_name] = annotator_data

    return annotator_overall_data


metrics = {}
for seed in range(max_seeds):
    metrics[f"seed{seed}"] = compute_metrics(f"../data/paper/human/arena_w_human_plus_ai_anns_ap_seed{seed}.json")

In [ ]:
# average metrics over seeds
metrics_avg = {}
for model in metrics[f"seed0"].keys():
    metrics_avg[model] = {}
    for annotator in metrics[f"seed0"][model].keys():
        metrics_avg[model][annotator] = {}
        metric_names = metrics[f"seed0"][model][annotator].keys()
        for metric in metric_names:
            values = [metrics[f"seed{seed}"][model][annotator][metric] for seed in range(max_seeds)]
            mean = np.mean(values)
            std = np.std(values)
            metrics_avg[model][annotator][metric] = {"mean": mean, "std": std, "formatted": f"{mean:.2f} \\textcolor{{gray}}{{±{std:.2f}}}", "formatted_emph": f"\\textbf{{{mean:.2f}}} \\textcolor{{gray}}{{±{std:.2f}}}"}

In [ ]:
import pandas as pd
import copy

annotator_overall_data = metrics_avg

def plot_ipynb_table(models = ['gpt-4o-mini', 'gpt-4.1-mini', 'gpt-5-mini', 'gemini-2.5-flash']):
    d = copy.deepcopy(annotator_overall_data)
    for model in models:
        for annotator in d[model].keys():
            for metric in d[model][annotator].keys():
                if isinstance(d[model][annotator][metric], dict):
                    d[model][annotator][metric] = d[model][annotator][metric]["mean"]
    cols = ['agreement_on_relevance', 'agreement_relevant'] #'prop_invalid']

    table_df = pd.concat(
        {model: pd.DataFrame(traits).T[cols] for model, traits in d.items() if model in models},
        axis=1
    )

    idx = pd.IndexSlice
    styler = table_df.style
    for metric in cols:
        styler = styler.highlight_max(axis=1, subset=idx[:, idx[:, metric]], props='font-color: white; background-color: blue')

    styler = styler.format(precision=2, na_rep='—')
    return styler

plot_ipynb_table()

In [ ]:
# create latex table

# Create a clean LaTeX table with minimal lines and bold top results
import pandas as pd
import numpy as np
import copy

def generate_latex_table(models: list[str], output_path: str, rename_models: dict[str, str] = {}):

    # Get the data
    d = copy.deepcopy(annotator_overall_data)
    cols = ['agreement_on_relevance', 'agreement_relevant']

    for model in models:
        assert model in d.keys(), f"Model {model} not found in data ({d.keys()})"

    if rename_models:
        for old_model, new_model in rename_models.items():
            if old_model in d.keys():
                d[new_model] = d[old_model]
                del d[old_model]
            if old_model in models:
                models[models.index(old_model)] = new_model


    table_df = pd.concat(
        {model: pd.DataFrame(traits).T[cols] for model, traits in d.items() if model in models},
        axis=1
    )

    # Find the maximum value for each row (principle) and each metric
    max_values = {}
    for principle in table_df.index:
        for metric in cols:
            max_val = table_df.loc[principle, (slice(None), metric)].apply(lambda x: x["mean"]).max()
            max_values[(principle, metric)] = max_val

    # Calculate min and mean values for each model and metric
    min_values = {}
    mean_values = {}
    for model in models:
        for metric in cols:
            values = table_df.loc[:, (model, metric)].dropna().apply(lambda x: x["mean"])
            min_values[(model, metric)] = values.min()
            mean_values[(model, metric)] = values.mean()

    # Find max values for min and mean rows
    min_max_values = {}
    mean_max_values = {}
    for metric in cols:
        min_vals = [min_values[(model, metric)] for model in models]
        mean_vals = [mean_values[(model, metric)] for model in models]
        min_max_values[metric] = max(min_vals)
        mean_max_values[metric] = max(mean_vals)

    # Create LaTeX table with bold formatting for maximum values
    latex_lines = []
    latex_lines.append("\\begin{minipage}[t]{1\\textwidth}")
    latex_lines.append("\\centering")
    latex_lines.append("\\sffamily")
    latex_lines.append("\\tablefontsize")

    # Create column specification with wrapping for first column
    n_models = len(models)
    col_spec = "p{3.8cm}" + "cc" * n_models
    latex_lines.append(f"\\begin{{tabular}}{{{col_spec}}}")
    latex_lines.append("\\toprule")

    # Header row
    header = "" #"Principle"
    for model in models:
        header += f" & \\multicolumn{{2}}{{c}}{{\\textbf{{{model}}}}}"
    latex_lines.append(header + " \\\\")

    # Subheader row
    subheader = "\\textbf{Trait}"
    for model in models:
        subheader += " & \\textit{Relevance} & \\textit{Choice}"
    latex_lines.append(subheader + " \\\\")
    latex_lines.append("\\midrule")

    # Data rows
    for principle in table_df.index:
        row = principle.replace("_", "\\_")  # Escape underscores for LaTeX

        for model in models:
            for metric in cols:
                value = table_df.loc[principle, (model, metric)]
                if pd.isna(value):
                    formatted_value = "—"
                else:
                    formatted_value = value["formatted"] #f"{value:.2f}"
                    # Bold if this is the maximum value for this principle and metric
                    if value["mean"] == max_values[(principle, metric)]:
                        formatted_value = value["formatted_emph"]

                row += f" & {formatted_value}"

        latex_lines.append(row + " \\\\")

    latex_lines.append("\\midrule")

    # Min row
    min_row = "\\textit{Min}"
    for model in models:
        for metric in cols:
            value = min_values[(model, metric)]
            formatted_value = f"{value:.2f}"
            # Bold if this is the maximum value for this metric in the min row
            if value == min_max_values[metric]:
                formatted_value = f"\\textbf{{{formatted_value}}}"
            min_row += f" & {formatted_value}"
    latex_lines.append(min_row + " \\\\")

    # Mean row
    mean_row = "\\textit{Mean}"
    for model in models:
        for metric in cols:
            value = mean_values[(model, metric)]
            formatted_value = f"{value:.2f}"
            # Bold if this is the maximum value for this metric in the mean row
            if value == mean_max_values[metric]:
                formatted_value = f"\\textbf{{{formatted_value}}}"
            mean_row += f" & {formatted_value}"
    latex_lines.append(mean_row + " \\\\")

    latex_lines.append("\\bottomrule")
    latex_lines.append("\\end{tabular}")
    latex_lines.append("\\end{minipage}")

    latex_table = "\n".join(latex_lines)

    # save to file
    with open(output_path, "w") as f:
        f.write(latex_table)

generate_latex_table(models = ['gpt-4o-mini', 'gpt-4.1-mini', 'gpt-5-mini', 'gemini-2.5-flash'], output_path = "output/tex/010_human_ai_agreement_table.tex")

generate_latex_table(models = ['gemini-2.5-flash-single', 'gemini-2.5-flash'], output_path = "output/tex/011_human_ai_agreement_table_single_vs_multi.tex", rename_models = {"gemini-2.5-flash": "Multi-vote", "gemini-2.5-flash-single": "Single-vote"})